# Step 6 — Affordance head debug

**Goal:** Train a small MLP that maps `concat(VLM vertex feature, verb embedding [, SAM3D global latent])` → per-vertex affordance score, and verify it learns on a single mesh.

**Prerequisites:**
- `outputs/notebooks/05_projection/vertex_semantic.pt` (run notebook 05 first)
- A mesh consistent with steps 2–5: `data/sample.glb` **or** set **`SAM3D_RUN_DIR`** in the next cell to notebook 10’s `RUN_DIR` (same as in notebooks **02** / **04** / **05**).
- **Optional SAM3D shape signal:** set **`SAM3D_RUN_DIR`** (reads `reconstruction/meta.json` + loads `paths.cache_root/sam3d/<stem>/global_latent.pt`), or set **`SAM3D_GLOBAL_LATENT_PATH`**, or **`SAM3D_OBJECT_STEM`**. If none resolve, the head uses VLM+verb only (`include_sam3d=False`).
- `affordancePrediction` conda env activated

**No 3DAffordSplat / real affordance labels needed for this notebook.** Pseudo-GT labels = cosine similarity of vertex VLM features to verb embedding (thresholded). This validates the architecture before wiring [3DAffordSplat](https://arxiv.org/abs/2504.11218) supervision (see `docs/data_strategy_3daffordsplat.md`).

**Pass checklist (end of notebook):**
- [ ] MLP forward pass produces `(V,)` logits
- [ ] Loss decreases monotonically over 200 steps on ≤500 vertices
- [ ] Predicted affordance heatmap on mesh is non-trivial (not all one colour)
- [ ] Predicted high-affordance vertices overlap with pseudo-GT high-similarity vertices

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run from the repo or notebooks/.")

from reconstruction.sam3d_wrapper import try_load_cached_global_latent
from utils.config import load_config, project_root, resolve_path

ROOT = project_root()
cfg = load_config()

# Optional: same RUN_DIR as notebook 10 (parent of reconstruction/mesh.glb).
SAM3D_RUN_DIR: Path | None = None
# Optional: direct path to global_latent.pt (overrides stem inference).
SAM3D_GLOBAL_LATENT_PATH: Path | None = None
# Optional: cache stem if you have no meta.json but latent exists under paths.cache_root/sam3d/<stem>/
SAM3D_OBJECT_STEM: str | None = None

if SAM3D_RUN_DIR is not None:
    from reconstruction.mesh_utils import cfg_with_sam3d_reconstruction

    cfg = cfg_with_sam3d_reconstruction(cfg, SAM3D_RUN_DIR)

sam3d_global_vec, sam3d_latent_path = try_load_cached_global_latent(
    cfg,
    sam3d_run_dir=SAM3D_RUN_DIR,
    object_stem=SAM3D_OBJECT_STEM,
    latent_path=SAM3D_GLOBAL_LATENT_PATH,
)
USE_SAM3D_LATENT = sam3d_global_vec is not None
if USE_SAM3D_LATENT:
    sam3d_global_vec = sam3d_global_vec.float().contiguous()
    print("SAM3D global latent:", sam3d_latent_path, "| shape", tuple(sam3d_global_vec.shape))
elif sam3d_latent_path is not None:
    print("SAM3D latent not found (MLP uses VLM+verb only):", sam3d_latent_path)
else:
    print(
        "SAM3D latent not resolved — set SAM3D_GLOBAL_LATENT_PATH, SAM3D_OBJECT_STEM, "
        "or SAM3D_RUN_DIR (with reconstruction/meta.json + cached global_latent.pt)."
    )

print("Config loaded. model config:", cfg.get("model"))
print("rendering.mesh_path:", cfg["rendering"]["mesh_path"])

## 1 — Load vertex VLM features from stage 04 cache

In [ ]:
import torch
import numpy as np

PROJ_CACHE = ROOT / "outputs" / "notebooks" / "05_projection" / "vertex_semantic.pt"
assert PROJ_CACHE.exists(), f"Missing: {PROJ_CACHE} — run notebook 05 first"

proj = torch.load(PROJ_CACHE, weights_only=False)
vlm_features   = proj["features"]           # (V, 512)
visible_mask   = torch.from_numpy(proj["visible_in_any_view"])  # (V,) bool

V, D = vlm_features.shape
print(f"Vertices: {V}  |  VLM dim: {D}  |  Visible: {visible_mask.sum().item()}")
assert D == cfg["model"]["vlm_dim"], f"Dim mismatch: got {D}, config says {cfg['model']['vlm_dim']}"

## 2 — Encode a verb with the frozen CLIP text encoder

In [ ]:
from vlm.vlm_wrapper import VLMWrapper
from vlm.text_encoder import encode_texts

VERB = "grasp"

vlm = VLMWrapper()
verb_emb = encode_texts(vlm, [VERB])[0]   # (512,)
print(f"Verb: '{VERB}'  |  embedding norm: {verb_emb.norm().item():.4f}")

## 3 — Build pseudo-GT labels

Cosine similarity of each vertex VLM feature to the verb embedding, min-max normalised to [0, 1].  
This is a reasonable proxy: vertices whose visual appearance aligns with "grasp" should score high.

In [ ]:
import matplotlib.pyplot as plt

v_norm = vlm_features / vlm_features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
t_norm = verb_emb / verb_emb.norm().clamp(min=1e-8)
cos_sim = (v_norm @ t_norm)              # (V,)

# Normalise to [0, 1] over visible vertices only
sim_vis = cos_sim[visible_mask]
lo, hi = sim_vis.min(), sim_vis.max()
pseudo_labels = torch.zeros(V)
pseudo_labels[visible_mask] = (sim_vis - lo) / (hi - lo + 1e-8)

plt.figure(figsize=(7, 3))
plt.hist(pseudo_labels[visible_mask].numpy(), bins=40, color="steelblue", edgecolor="k", linewidth=0.3)
plt.title(f"Pseudo-GT label distribution — verb: '{VERB}'")
plt.xlabel("label value")
plt.ylabel("# vertices")
plt.tight_layout()
plt.show()

print(f"Label stats — min: {pseudo_labels[visible_mask].min():.3f}  max: {pseudo_labels[visible_mask].max():.3f}  mean: {pseudo_labels[visible_mask].mean():.3f}")

## 4 — Build the MLP and verify forward pass

In [ ]:
from models.mlp_head import build_affordance_mlp

model = build_affordance_mlp(cfg, include_sam3d=USE_SAM3D_LATENT)
if USE_SAM3D_LATENT:
    assert sam3d_global_vec.numel() == model.cfg.sam3d_dim, (
        f"Latent length {sam3d_global_vec.numel()} != model.sam3d_dim {model.cfg.sam3d_dim}"
    )
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Smoke-test forward pass
with torch.no_grad():
    dummy_logits = model(
        vlm_features,
        verb_emb,
        sam3d_global_vec if USE_SAM3D_LATENT else None,
    )
assert dummy_logits.shape == (V,), f"Expected ({V},), got {dummy_logits.shape}"
print(f"Forward pass OK — output shape: {dummy_logits.shape}")

## 5 — Overfit on a small subset (≤500 visible vertices)

In [ ]:
SUBSET = 500
N_STEPS = 200
LR = cfg["training"]["learning_rate"]

from training.affordance_fit import fit_affordance_mlp_simple

losses = fit_affordance_mlp_simple(
    model,
    vlm_features,
    verb_emb,
    pseudo_labels,
    vertex_mask=visible_mask,
    sam3d_global=sam3d_global_vec if USE_SAM3D_LATENT else None,
    max_steps=N_STEPS,
    lr=LR,
    subset_size=SUBSET,
)

print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f}")

## 6 — Loss curve

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(losses, linewidth=1.2)
plt.title(f"Training loss — verb: '{VERB}', subset: {SUBSET} vertices")
plt.xlabel("step")
plt.ylabel("BCE loss")
plt.tight_layout()
plt.show()

assert losses[-1] < losses[0], "Loss did not decrease — training failed"

## 7 — Predict on all visible vertices and color the mesh

In [ ]:
import matplotlib.cm as cm
from datasets.mesh_loading import load_mesh, normalize_mesh

model.eval()
with torch.no_grad():
    all_logits = model(
        vlm_features,
        verb_emb,
        sam3d_global_vec if USE_SAM3D_LATENT else None,
    )  # (V,)
    all_probs = torch.sigmoid(all_logits)  # (V,)

# Build vertex colors: invisible vertices grey, visible colored by predicted score
scores = all_probs.numpy()
vertex_colors = np.full((V, 3), 180, dtype=np.uint8)  # grey default
vis_np = visible_mask.numpy()
cmap = cm.get_cmap("RdYlGn")
vertex_colors[vis_np] = (cmap(scores[vis_np])[:, :3] * 255).astype(np.uint8)

# Load mesh and apply colors
mesh_path = resolve_path(cfg["rendering"]["mesh_path"], root=ROOT)
assert mesh_path.is_file(), f"Missing mesh: {mesh_path}"
mesh = normalize_mesh(load_mesh(mesh_path))

import trimesh
tm = trimesh.Trimesh(
    vertices=mesh.vertices,
    faces=mesh.faces,
    vertex_colors=vertex_colors,
    process=False,
)

print(f"Predicted scores — min: {scores[vis_np].min():.3f}  max: {scores[vis_np].max():.3f}  mean: {scores[vis_np].mean():.3f}")
assert scores[vis_np].max() > 0.1 and scores[vis_np].min() < 0.9, "Scores are degenerate (all same value)"

scene = tm.scene()
scene.show()

## 8 — Compare predicted vs pseudo-GT (top-K overlap)

In [ ]:
K = 200
vis_scores  = all_probs[visible_mask].numpy()
vis_labels  = pseudo_labels[visible_mask].numpy()

top_pred = set(np.argsort(vis_scores)[-K:])
top_gt   = set(np.argsort(vis_labels)[-K:])
overlap  = len(top_pred & top_gt)
iou      = overlap / len(top_pred | top_gt)

print(f"Top-{K} overlap: {overlap}/{K}  |  IoU: {iou:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, vals, title in zip(axes, [vis_scores, vis_labels], ["Predicted (sigmoid)", "Pseudo-GT (cos sim)"]):
    ax.hist(vals, bins=40, color="steelblue", edgecolor="k", linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel("score")
plt.tight_layout()
plt.show()

## 9 — Save model checkpoint

In [ ]:
OUT_DIR = ROOT / "outputs" / "notebooks" / "06_affordance_head"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = OUT_DIR / f"mlp_{VERB.replace(' ', '_')}.pt"
torch.save(
    {
        "model_state": model.state_dict(),
        "config": model.cfg,
        "verb": VERB,
        "final_loss": losses[-1],
        "use_sam3d_latent": USE_SAM3D_LATENT,
        "sam3d_latent_path": str(sam3d_latent_path) if sam3d_latent_path is not None else None,
    },
    ckpt_path,
)
print(f"Checkpoint saved: {ckpt_path}")

## Pass checklist

Run this cell last. All assertions must pass.

In [ ]:
checks = [
    (dummy_logits.shape == (V,),             "MLP produces (V,) logits"),
    (losses[-1] < losses[0],                 "Loss decreased over training"),
    (scores[vis_np].max() > scores[vis_np].min() + 0.05,
                                             "Predicted scores are non-trivial (not all same)"),
    (ckpt_path.exists(),                     "Checkpoint saved"),
]

all_passed = True
for ok, desc in checks:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {desc}")
    if not ok:
        all_passed = False

assert all_passed, "One or more checks failed — see above"
print("\nAll checks passed. Step 6 complete.")